In [0]:
# DEPLOYABLE TEMPLATE | Databricks Python notebook
from pyspark.sql import functions as F
p = spark.table('dcp_synth.`00_config`.synthetic_parameters').first()
root, seed = p.landing_root, p.deterministic_seed
run_id = f'run_{p.fixed_generated_at:%Y%m%dT%H%M%S}'
patients = (spark.range(1, p.patient_count + 1)
 .select(F.format_string('SYN-P-%04d', F.col('id')).alias('patient_id'),
 F.format_string('Synthetic-%04d', F.col('id')).alias('display_name'),
 F.expr("date_add(date'1955-01-01', CAST(pmod(xxhash64(id, 42), 18000) AS INT))").alias('birth_date'),
 F.element_at(F.array(*[F.lit(x) for x in ['DE','CH','FR','IT']]), F.expr('CAST(1+pmod(id, 4) AS INT)')).alias('country'),
 F.lit('SYNTHETIC').alias('data_class'), F.lit(run_id).alias('_run_id')))
(patients.coalesce(1).write.mode('overwrite').option('header',True)
 .csv(f'{root}/SRC-11/patient_master/{run_id}'))
# CGM uses Parquet to avoid impractically large text files while preserving file-first ingestion.
times = spark.sql(f"SELECT explode(sequence(timestamp'{p.history_start_date} 00:00:00', timestamp'{p.history_end_date} 23:55:00', interval 5 minutes)) event_time")
cgm=(patients.select('patient_id').crossJoin(times)
 .withColumn('reading_id',F.sha2(F.concat_ws('|','patient_id',F.col('event_time').cast('string')),256))
 .withColumn('glucose_mmol_l',F.round(F.greatest(F.lit(2.2),F.least(F.lit(22.0),F.lit(6.8)+(F.pmod(F.xxhash64('patient_id','event_time',F.lit(seed)),F.lit(1000))/1000-.5)*2.4)),1))
 .withColumn('source_id',F.lit('SRC-01')).withColumn('_run_id',F.lit(run_id)))
(cgm.write.mode('overwrite').partitionBy('_run_id').parquet(f'{root}/SRC-01/cgm_readings'))
# Smaller event streams are JSON/NDJSON.
(patients.select('patient_id',F.lit('SRC-16').alias('source_id'),F.lit(run_id).alias('_run_id'))
 .write.mode('overwrite').json(f'{root}/SRC-16/device_events/{run_id}'))